In [1]:

# Install dependencies
!pip install kagglehub tensorflow numpy matplotlib --quiet

import kagglehub
import os, glob, shutil
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG19
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.optimizers import Adam

# -------------------------
# 1. Download Dataset
# -------------------------
path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
print("Path to dataset files:", path)

# -------------------------
# 2. Detect dataset structure and split 80/20
# -------------------------
# Detect main dataset folder (some versions have /Training and /Testing)
if os.path.exists(os.path.join(path, "Training")):
    dataset_path = os.path.join(path, "Training")
elif os.path.exists(os.path.join(path, "train")):
    dataset_path = os.path.join(path, "train")
else:
    dataset_path = path

print(f"Detected dataset folder: {dataset_path}")

# Prepare split directories
base_dir = "/content/brain_tumor_split"
train_dir = os.path.join(base_dir, "train")
test_dir = os.path.join(base_dir, "test")
os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

# Split class folders 80/20
folders = [f for f in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, f))]
for folder in folders:
    img_paths = glob.glob(os.path.join(dataset_path, folder, "*"))
    train_files, test_files = train_test_split(img_paths, test_size=0.2, random_state=42)

    os.makedirs(os.path.join(train_dir, folder), exist_ok=True)
    os.makedirs(os.path.join(test_dir, folder), exist_ok=True)

    for f in train_files:
        if os.path.isfile(f):
            shutil.copy(f, os.path.join(train_dir, folder))
    for f in test_files:
        if os.path.isfile(f):
            shutil.copy(f, os.path.join(test_dir, folder))

print("✅ Dataset split completed successfully: 80% train / 20% test")

# -------------------------
# 3. Image Data Generators (Resizing + Normalization + Augmentation)
# -------------------------
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

# -------------------------
# 4. Build VGG19 Model
# -------------------------
base_model = VGG19(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze base layers
for layer in base_model.layers:
    layer.trainable = False

# Add custom head
x = Flatten()(base_model.output)
x = Dense(512, activation='relu')(x)
x = Dropout(0.25)(x)
output = Dense(train_generator.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

print(model.summary())

# -------------------------
# 5. Train the Model
# -------------------------
history = model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=10,       # you can increase to 20–30 for higher accuracy
    verbose=1
)

# -------------------------
# 6. Evaluate the Model
# -------------------------
loss, acc = model.evaluate(test_generator)
print(f"\n✅ Test Accuracy: {acc:.4f}")


Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.
Path to dataset files: /kaggle/input/brain-tumor-mri-dataset
Detected dataset folder: /kaggle/input/brain-tumor-mri-dataset/Training
✅ Dataset split completed successfully: 80% train / 20% test
Found 4568 images belonging to 4 classes.
Found 1144 images belonging to 4 classes.


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv4 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv4 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv4 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │    12,845,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             

 Total params: 32,872,004 (125.40 MB)

 Trainable params: 12,847,620 (49.01 MB)

 Non-trainable params: 20,024,384 (76.39 MB)

None


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
143/143 ━━━━━━━━━━━━━━━━━━━━ 97s 580ms/step - accuracy: 0.6215 - loss: 0.9468 - val_accuracy: 0.8077 - val_loss: 0.4704
Epoch 2/10
143/143 ━━━━━━━━━━━━━━━━━━━━ 70s 490ms/step - accuracy: 0.8083 - loss: 0.4819 - val_accuracy: 0.8575 - val_loss: 0.3819
Epoch 3/10
143/143 ━━━━━━━━━━━━━━━━━━━━ 70s 490ms/step - accuracy: 0.8436 - loss: 0.3963 - val_accuracy: 0.8811 - val_loss: 0.3331
Epoch 4/10
143/143 ━━━━━━━━━━━━━━━━━━━━ 70s 489ms/step - accuracy: 0.8609 - loss: 0.3736 - val_accuracy: 0.8776 - val_loss: 0.3183
Epoch 5/10
143/143 ━━━━━━━━━━━━━━━━━━━━ 72s 501ms/step - accuracy: 0.8718 - loss: 0.3429 - val_accuracy: 0.9065 - val_loss: 0.2746
Epoch 6/10
143/143 ━━━━━━━━━━━━━━━━━━━━ 70s 490ms/step - accuracy: 0.8902 - loss: 0.2966 - val_accuracy: 0.8733 - val_loss: 0.3217
Epoch 7/10
143/143 ━━━━━━━━━━━━━━━━━━━━ 70s 487ms/step - accuracy: 0.8840 - loss: 0.3066 - val_accuracy: 0.8846 - val_loss: 0.3160
Epoch 8/10
143/143 ━━━━━━━━━━━━━━━━━━━━ 70s 488ms/step - accuracy: 0.8875 - loss: 0